# Causality

This notebook contains two connected but deliberately separate experiments.

1. **Causal association and retrieval** asks whether a two-hop question can identify the one independent edit that changes its answer. Local mode compares the owner edit with controlled negatives. Global mode retrieves the owner from increasingly large edit banks. Only the causal prompt and candidate edit `{subject, prompt, answer}` are visible to the scorer; hop IDs and dependency labels are hidden evaluation metadata.
2. **Causal intervention diagnostics** assume that retrieval is already correct and provide the known owner edit directly. Hidden-state oracle routing tests whether Mnemo's stored edit vector can produce the final two-hop answer. Context diagnostics prepend the known fact as text. Prerequisite diagnostics test whether the model can recover the unchanged hop required to finish the chain.

The data contains independent forward-only `A -> B -> C` chains from UniEdit. The edit is at source position 0 when it changes `A -> B`, making the edited answer an intermediate connector. It is at source position 1 when it changes `B -> C`, making the edited answer the final requested answer. Every retrieval and intervention result must therefore be read separately by source position.

Retrieval success does not prove causal success. A scorer can retrieve the correct edit while an answer-injection vector emits only the intermediate connector. Conversely, oracle intervention can work when retrieval fails. The causality split also contains no no-edit/locality prompts, so its association threshold is not a safe router activation threshold.

## Environment

In [ ]:
import gc
import hashlib
import json
import math
import random
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from contextlib import nullcontext, redirect_stderr
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_from_disk
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.editors.editor_utils import EditRequest
from src.editors.mnemo import Mnemo
from src.utils.devices import get_model_device

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "causality"
DATASET_SPLIT = "data"
LOCAL_DATASET_PATH = PROJECT_ROOT / "data/datasets/uniedit/final"
OUTPUT_DIR = PROJECT_ROOT / "data/causality"
EMBEDDING_MODEL_REPO = "BAAI/bge-m3"
SCORER_ORDER = (
    "bm25_prompt",
    "bge_prompt",
    "bge_subject_max",
    "bge_composed",
)
SCORER_LABELS = {
    "bm25_prompt": "BM25: edit prompt",
    "bge_prompt": "BGE-M3: edit prompt",
    "bge_subject_max": "BGE-M3: max(prompt, subject)",
    "bge_composed": "BGE-M3: subject + edit + answer",
}

print(f"local_dataset={LOCAL_DATASET_PATH}")
print(f"output_dir={OUTPUT_DIR}")

## Part I: Causal Association And Retrieval

This stage never edits or generates with the language model. It measures whether lexical or embedding scorers can associate each causal question with its responsible edit. Local metrics answer whether the owner beats carefully selected negatives; global Recall@k and MRR answer whether the owner remains discoverable as unrelated edits enter the bank. Source-position breakdowns reveal whether the edited subject is directly visible in the question or hidden behind the unchanged first hop.

## Dataset Contract

The retained rows expose only information available to a training-free retriever while preserving dependency structure as hidden labels for deterministic evaluation.

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(str(text).split()).casefold()


def prompt_key(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    normalized = "".join(
        character
        if character.isalnum() or unicodedata.category(character).startswith("S")
        else " "
        for character in normalized
    )
    return " ".join(normalized.split())


def answer_keys(values: list[str]) -> set[str]:
    keys = set()
    for value in values:
        normalized = unicodedata.normalize("NFKC", str(value)).casefold()
        words = tuple(re.findall(r"\w+", normalized))
        if words:
            keys.add("".join(words))
            if len(words) > 1 and words[0] in {"a", "an", "the"}:
                keys.add("".join(words[1:]))
        compact = "".join(
            character
            for character in normalized
            if character.isalnum()
            or unicodedata.category(character).startswith("S")
        )
        if compact:
            keys.add(compact)
    return keys


def owner_id(edit: dict) -> str:
    return f"{edit['subject']['id']}|{edit['relation']['id']}"


def runtime_edit(edit: dict) -> dict[str, str]:
    return {
        "subject": edit["subject"]["str"],
        "prompt": edit["prompt"],
        "answer": edit["answers"][0],
    }


def runtime_query(causality: dict) -> str:
    return causality["prompt"]


def load_causality_dataset(dataset_source: str):
    if dataset_source == "local":
        dataset = load_from_disk(str(LOCAL_DATASET_PATH))
        return dataset["causality"]
    if dataset_source == "huggingface":
        return load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def source_record_count(dataset, record_limit: int | None) -> int:
    if record_limit is None:
        return len(dataset)
    if int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    return min(int(record_limit), len(dataset))


def build_causality_records(dataset, record_limit: int | None) -> list[dict]:
    records = []
    seen_owners = set()
    seen_edit_prompts = set()
    seen_queries = set()
    for source_index in range(source_record_count(dataset, record_limit)):
        source = dataset[source_index]
        edit = source["edit"]
        current_owner = owner_id(edit)
        edit_prompt_key = prompt_key(edit["prompt"])
        if current_owner in seen_owners:
            raise ValueError(f"Duplicate edit owner at row {source_index}: {current_owner}")
        if edit_prompt_key in seen_edit_prompts:
            raise ValueError(f"Duplicate canonical edit prompt at row {source_index}.")
        if not edit["answers"] or not edit["subject"]["str"]:
            raise ValueError(f"Incomplete edit at row {source_index}.")
        seen_owners.add(current_owner)
        seen_edit_prompts.add(edit_prompt_key)

        queries = []
        for query_index, causal in enumerate(source["causality"]):
            query_prompt = runtime_query(causal)
            query_prompt_key = prompt_key(query_prompt)
            if not query_prompt_key or query_prompt_key in seen_queries:
                raise ValueError(f"Missing or repeated causal prompt at row {source_index}.")
            if query_prompt_key in seen_edit_prompts:
                raise ValueError(f"Causal prompt collides with an edit prompt at row {source_index}.")
            if not causal["answers"] or len(causal["hops"]) != 2:
                raise ValueError(f"Incomplete causal evidence at row {source_index}.")
            position = int(causal["source_edit_position"])
            if position not in {0, 1}:
                raise ValueError(f"Invalid source edit position at row {source_index}.")
            hops = causal["hops"]
            if sum(bool(hop["is_source_edit"]) for hop in hops) != 1:
                raise ValueError(f"Causal query has multiple source edits at row {source_index}.")
            if not hops[position]["is_source_edit"]:
                raise ValueError(f"Source position disagrees with hop labels at row {source_index}.")
            if any(
                hop["direction"] != "forward" or not hop["single_value"]
                for hop in hops
            ):
                raise ValueError(f"Non-forward or multi-valued hop at row {source_index}.")
            if hops[0]["tail"]["id"] != hops[1]["head"]["id"]:
                raise ValueError(f"Broken two-hop entity continuity at row {source_index}.")
            if (
                hops[position]["head"]["id"] != edit["subject"]["id"]
                or hops[position]["relation"]["id"] != edit["relation"]["id"]
            ):
                raise ValueError(f"Source hop disagrees with its edit at row {source_index}.")
            edit_answer_keys = answer_keys(list(edit["answers"]))
            if not edit_answer_keys & answer_keys(list(hops[position]["answers"])):
                raise ValueError(f"Source hop answer disagrees with edit at row {source_index}.")
            expected_dependency = [{
                "subject_id": edit["subject"]["id"],
                "relation_id": edit["relation"]["id"],
            }]
            if list(causal["dependency_edit_keys"]) != expected_dependency:
                raise ValueError(f"Causal query is not single-dependency at row {source_index}.")
            causal_answer_keys = answer_keys(list(causal["answers"]))
            if position == 0 and edit_answer_keys & causal_answer_keys:
                raise ValueError(f"Position-0 query can bypass hop two at row {source_index}.")
            if position == 1 and not edit_answer_keys & causal_answer_keys:
                raise ValueError(f"Position-1 query answer disagrees with edit at row {source_index}.")
            seen_queries.add(query_prompt_key)
            query_digest = hashlib.sha256(query_prompt_key.encode("utf-8")).hexdigest()[:12]
            queries.append({
                "query_id": f"{current_owner}|{query_index}|{query_digest}",
                "prompt": query_prompt,
                "answers": list(causal["answers"]),
                "source_edit_position": position,
                "origin": causal["origin"],
                "hidden_hops": list(hops),
            })
        if not queries:
            raise ValueError(f"Edit row {source_index} has no causal query.")
        records.append({
            "source_index": source_index,
            "owner_id": current_owner,
            "edit": runtime_edit(edit),
            "edit_answers": list(edit["answers"]),
            "queries": queries,
        })

    all_edit_keys = {prompt_key(record["edit"]["prompt"]) for record in records}
    all_query_keys = {
        prompt_key(query["prompt"])
        for record in records
        for query in record["queries"]
    }
    if all_edit_keys & all_query_keys:
        raise ValueError("Final causal prompts overlap canonical edit prompts.")
    return records


def causality_prompt_fingerprint(records: list[dict]) -> str:
    digest = hashlib.sha256()
    for record in records:
        payload = {
            "owner_id": record["owner_id"],
            "edit": record["edit"],
            "queries": [
                {
                    "prompt": query["prompt"],
                    "answers": query["answers"],
                    "source_edit_position": query["source_edit_position"],
                    "origin": query["origin"],
                }
                for query in record["queries"]
            ],
        }
        encoded = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def causality_dataset_metadata(dataset, records: list[dict], dataset_source: str) -> dict:
    metadata = {
        "source": dataset_source,
        "rows": len(records),
        "queries": sum(len(record["queries"]) for record in records),
        "fingerprint": getattr(dataset, "_fingerprint", None),
        "prompt_fingerprint": causality_prompt_fingerprint(records),
        "policy": {
            "independent_edits": True,
            "forward_only": True,
            "hop_count": 2,
            "outer_double_questions": False,
            "nested_single_paths": True,
        },
    }
    if dataset_source == "local":
        metadata["dataset_id"] = "uniedit/final:causality"
    else:
        metadata.update({
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
        })
    return metadata

## Scorers

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model():
    from FlagEmbedding import BGEM3FlagModel

    model = BGEM3FlagModel(
        EMBEDDING_MODEL_REPO,
        use_fp16=embedding_compute_dtype() == "float16",
        devices=embedding_device(),
    )
    print(
        f"loaded_embedding_model={EMBEDDING_MODEL_REPO} "
        f"device={embedding_device()} compute_dtype={embedding_compute_dtype()}"
    )
    return model


def encode_prompts(model, prompts: list[str]) -> np.ndarray:
    if not prompts:
        raise ValueError("Cannot encode an empty prompt list.")
    with redirect_stderr(StringIO()):
        encoded = model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
    embeddings = np.asarray(encoded["dense_vecs"], dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
    return embeddings


def release_embedding_model(model) -> None:
    underlying = getattr(model, "model", None)
    if underlying is not None and hasattr(underlying, "to"):
        underlying.to("cpu")
    model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def lexical_tokens(text: str) -> tuple[str, ...]:
    return tuple(re.findall(r"\w+", normalize_text(text)))


class Bm25Index:
    def __init__(self, documents: list[str], k1: float = 1.5, b: float = 0.75):
        if not documents:
            raise ValueError("BM25 needs at least one document.")
        self.k1 = float(k1)
        self.b = float(b)
        self.document_count = len(documents)
        self.document_lengths = np.asarray(
            [len(lexical_tokens(document)) for document in documents],
            dtype=np.float32,
        )
        self.average_length = max(float(self.document_lengths.mean()), 1.0)
        document_frequencies = {}
        postings = {}
        for document_index, document in enumerate(documents):
            frequencies = {}
            for token in lexical_tokens(document):
                frequencies[token] = frequencies.get(token, 0) + 1
            for token, frequency in frequencies.items():
                postings.setdefault(token, []).append((document_index, frequency))
                document_frequencies[token] = document_frequencies.get(token, 0) + 1
        self.postings = postings
        self.idf = {
            token: math.log(
                1.0
                + (self.document_count - frequency + 0.5)
                / (frequency + 0.5)
            )
            for token, frequency in document_frequencies.items()
        }

    def score(self, query: str) -> np.ndarray:
        scores = np.zeros(self.document_count, dtype=np.float32)
        for token in set(lexical_tokens(query)):
            if token not in self.postings:
                continue
            idf = self.idf[token]
            for document_index, frequency in self.postings[token]:
                length = self.document_lengths[document_index]
                denominator = frequency + self.k1 * (
                    1.0 - self.b + self.b * length / self.average_length
                )
                scores[document_index] += idf * (
                    frequency * (self.k1 + 1.0) / denominator
                )
        return scores


def composed_edit_text(edit: dict[str, str]) -> str:
    return (
        f"Subject: {edit['subject']}\n"
        f"Edit: {edit['prompt']}\n"
        f"Answer: {edit['answer']}"
    )

## Deterministic Cohorts

In [ ]:
def seeded_rng(random_seed: int, *parts) -> np.random.Generator:
    payload = "|".join(str(part) for part in (random_seed, *parts)).encode("utf-8")
    seed = int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")
    return np.random.default_rng(seed)


def owner_is_calibration(owner: str, random_seed: int, fraction: float) -> bool:
    digest = hashlib.sha256(f"{random_seed}|split|{owner}".encode("utf-8")).digest()
    value = int.from_bytes(digest[:8], "big") / 2**64
    return value < fraction


def flatten_queries(records: list[dict]) -> list[dict]:
    queries = []
    for owner_index, record in enumerate(records):
        for query in record["queries"]:
            queries.append({
                **query,
                "owner_index": owner_index,
                "owner_id": record["owner_id"],
                "expected_edit": record["edit"],
            })
    return queries


def sample_queries(
    queries: list[dict],
    limit: int | None,
    random_seed: int,
    label: str,
) -> list[dict]:
    if limit is None or len(queries) <= int(limit):
        return list(queries)
    if int(limit) <= 0:
        raise ValueError("Query limits must be positive when set.")
    rng = seeded_rng(random_seed, label)
    indices = np.sort(rng.choice(len(queries), size=int(limit), replace=False))
    return [queries[int(index)] for index in indices]


def top_lexical_negatives(
    scores: np.ndarray,
    positive_index: int,
    limit: int,
) -> list[int]:
    candidate_count = min(len(scores) - 1, max(limit * 4, 64))
    if candidate_count <= 0:
        return []
    working = scores.copy()
    working[positive_index] = -np.inf
    if candidate_count < len(working):
        indices = np.argpartition(-working, candidate_count - 1)[:candidate_count]
    else:
        indices = np.arange(len(working))
    return [
        int(index)
        for index in indices[np.argsort(-working[indices])]
        if int(index) != positive_index
    ][:limit]


def fill_negative_indices(
    preferred: list[int],
    positive_index: int,
    record_count: int,
    negative_count: int,
    rng: np.random.Generator,
) -> list[int]:
    selected = []
    seen = {positive_index}
    for index in preferred:
        index = int(index)
        if index not in seen:
            seen.add(index)
            selected.append(index)
        if len(selected) == negative_count:
            return selected
    remaining = np.asarray(
        [index for index in range(record_count) if index not in seen],
        dtype=np.int64,
    )
    needed = negative_count - len(selected)
    if needed > len(remaining):
        raise ValueError("Not enough unique edits for the requested negatives.")
    if needed:
        chosen = rng.choice(remaining, size=needed, replace=False)
        selected.extend(int(index) for index in chosen)
    return selected


def build_local_cohorts(
    records: list[dict],
    queries: list[dict],
    bm25: Bm25Index,
    negative_count: int,
    random_seed: int,
    calibration_fraction: float,
) -> list[dict]:
    if negative_count <= 0 or negative_count >= len(records):
        raise ValueError("local_negative_count must be between 1 and bank size - 1.")
    subject_indices = {}
    answer_indices = {}
    for index, record in enumerate(records):
        subject_indices.setdefault(prompt_key(record["edit"]["subject"]), []).append(index)
        answer_indices.setdefault(prompt_key(record["edit"]["answer"]), []).append(index)

    cohorts = []
    for query in tqdm(queries, desc="Build local causal cohorts"):
        positive = int(query["owner_index"])
        lexical_scores = bm25.score(query["prompt"])
        random_rng = seeded_rng(random_seed, "local", "random", query["query_id"])
        random_negatives = fill_negative_indices(
            [], positive, len(records), negative_count, random_rng
        )

        edit = records[positive]["edit"]
        preferred = []
        preferred.extend(subject_indices.get(prompt_key(edit["subject"]), []))
        preferred.extend(answer_indices.get(prompt_key(edit["answer"]), []))
        preferred.extend(top_lexical_negatives(lexical_scores, positive, negative_count))
        challenge_rng = seeded_rng(random_seed, "local", "challenge", query["query_id"])
        challenge_negatives = fill_negative_indices(
            preferred, positive, len(records), negative_count, challenge_rng
        )

        for cohort_name, negatives in (
            ("random", random_negatives),
            ("challenge", challenge_negatives),
        ):
            candidate_indices = [positive, *negatives]
            cohorts.append({
                "cohort": cohort_name,
                "query": query,
                "candidate_indices": candidate_indices,
                "bm25_scores": lexical_scores[candidate_indices],
                "calibration": owner_is_calibration(
                    query["owner_id"], random_seed, calibration_fraction
                ),
            })
    return cohorts


def resolve_global_bank_sizes(
    requested_sizes: list[int | None],
    universe_size: int,
) -> list[tuple[int | None, int]]:
    resolved = []
    seen = set()
    for requested in requested_sizes:
        actual = universe_size if requested is None else int(requested)
        if actual < 2:
            raise ValueError("Every global bank must contain at least two edits.")
        if actual > universe_size:
            print(f"skipped_bank_size={actual} universe_size={universe_size}")
            continue
        key = (requested is None, actual)
        if key not in seen:
            seen.add(key)
            resolved.append((requested, actual))
    if not resolved:
        raise ValueError("No configured global bank size fits the dataset.")
    return sorted(resolved, key=lambda item: item[1])


def build_global_cohorts(
    records: list[dict],
    bank_sizes: list[int | None],
    repeats: int,
    query_owner_count: int,
    query_limit: int | None,
    random_seed: int,
) -> list[dict]:
    if repeats <= 0:
        raise ValueError("global_repeats must be positive.")
    resolved_sizes = resolve_global_bank_sizes(bank_sizes, len(records))
    smallest_bank = resolved_sizes[0][1]
    owner_count = min(int(query_owner_count), smallest_bank)
    if owner_count < 2:
        raise ValueError("global_query_owner_count must retain at least two owners.")
    cohorts = []
    for repeat_index in range(repeats):
        rng = seeded_rng(random_seed, "global", repeat_index)
        permutation = rng.permutation(len(records)).astype(np.int64)
        query_owners = set(int(index) for index in permutation[:owner_count])
        query_pool = [
            query
            for query in flatten_queries(records)
            if int(query["owner_index"]) in query_owners
        ]
        queries = sample_queries(
            query_pool,
            query_limit,
            random_seed,
            f"global-repeat-{repeat_index}",
        )
        cohorts.append({
            "repeat_index": repeat_index,
            "permutation": permutation,
            "query_owner_count": owner_count,
            "queries": queries,
            "bank_sizes": resolved_sizes,
        })
    return cohorts


def cohort_fingerprint(payload) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

## Metrics

In [ ]:
def rounded(value: float) -> float:
    return round(float(value), 8)


def rank_from_scores(positive_score: float, negative_scores: np.ndarray) -> int:
    return 1 + int(np.sum(negative_scores >= positive_score))


def local_observation_metrics(observations: list[dict]) -> dict:
    if not observations:
        return {"count": 0}
    auc_values = []
    ranks = []
    gaps = []
    wins = ties = losses = 0
    for observation in observations:
        positive = float(observation["positive_score"])
        negatives = np.asarray(observation["negative_scores"], dtype=np.float32)
        auc_values.append(float(np.mean((positive > negatives) + 0.5 * (positive == negatives))))
        rank = rank_from_scores(positive, negatives)
        ranks.append(rank)
        best_negative = float(negatives.max())
        gap = positive - best_negative
        gaps.append(gap)
        wins += gap > 0
        ties += gap == 0
        losses += gap < 0
    ranks = np.asarray(ranks, dtype=np.float32)
    gaps = np.asarray(gaps, dtype=np.float32)
    return {
        "count": len(observations),
        "pairwise_auc": rounded(np.mean(auc_values)),
        "top1_accuracy": rounded(np.mean(ranks == 1)),
        "mrr": rounded(np.mean(1.0 / ranks)),
        "mean_best_negative_gap": rounded(gaps.mean()),
        "gap_std": rounded(gaps.std()),
        "wins": int(wins),
        "ties": int(ties),
        "losses": int(losses),
    }


def fit_balanced_threshold(positive_scores: np.ndarray, negative_scores: np.ndarray) -> float:
    scores = np.concatenate([positive_scores, negative_scores]).astype(np.float64)
    labels = np.concatenate([
        np.ones(len(positive_scores), dtype=np.int8),
        np.zeros(len(negative_scores), dtype=np.int8),
    ])
    order = np.argsort(-scores, kind="mergesort")
    scores = scores[order]
    labels = labels[order]
    unique_starts = np.r_[0, np.flatnonzero(scores[1:] != scores[:-1]) + 1]
    positives = np.add.reduceat(labels, unique_starts)
    group_sizes = np.diff(np.r_[unique_starts, len(scores)])
    negatives = group_sizes - positives
    true_positives = np.cumsum(positives)
    false_positives = np.cumsum(negatives)
    positive_total = max(int(labels.sum()), 1)
    negative_total = max(int((labels == 0).sum()), 1)
    balanced = 0.5 * (
        true_positives / positive_total
        + 1.0 - false_positives / negative_total
    )
    best = int(np.argmax(balanced))
    return float(scores[unique_starts[best]])


def threshold_metrics(observations: list[dict]) -> dict:
    calibration = [item for item in observations if item["calibration"]]
    evaluation = [item for item in observations if not item["calibration"]]
    if not calibration or not evaluation:
        return {"available": False}
    calibration_positive = np.asarray(
        [item["positive_score"] for item in calibration], dtype=np.float32
    )
    calibration_negative = np.concatenate([
        np.asarray(item["negative_scores"], dtype=np.float32)
        for item in calibration
    ])
    threshold = fit_balanced_threshold(calibration_positive, calibration_negative)
    evaluation_positive = np.asarray(
        [item["positive_score"] for item in evaluation], dtype=np.float32
    )
    evaluation_negative = np.concatenate([
        np.asarray(item["negative_scores"], dtype=np.float32)
        for item in evaluation
    ])
    true_positive = int(np.sum(evaluation_positive >= threshold))
    false_positive = int(np.sum(evaluation_negative >= threshold))
    return {
        "available": True,
        "threshold": rounded(threshold),
        "calibration_queries": len(calibration),
        "evaluation_queries": len(evaluation),
        "evaluation_positive_recall": rounded(true_positive / len(evaluation_positive)),
        "evaluation_negative_false_positive_rate": rounded(
            false_positive / len(evaluation_negative)
        ),
        "evaluation_precision": rounded(
            true_positive / max(true_positive + false_positive, 1)
        ),
    }


def global_metrics(scores: np.ndarray, expected_positions: np.ndarray) -> dict:
    row_indices = np.arange(len(scores))
    correct = scores[row_indices, expected_positions]
    greater_or_equal = scores >= correct[:, None]
    greater_or_equal[row_indices, expected_positions] = False
    ranks = 1 + greater_or_equal.sum(axis=1)
    masked = scores.copy()
    masked[row_indices, expected_positions] = -np.inf
    best_negative = masked.max(axis=1)
    margins = correct - best_negative
    return {
        "count": len(scores),
        "recall_at_1": rounded(np.mean(ranks <= 1)),
        "recall_at_5": rounded(np.mean(ranks <= 5)),
        "recall_at_10": rounded(np.mean(ranks <= 10)),
        "recall_at_20": rounded(np.mean(ranks <= 20)),
        "recall_at_50": rounded(np.mean(ranks <= 50)),
        "mrr": rounded(np.mean(1.0 / ranks)),
        "rank_p50": rounded(np.percentile(ranks, 50)),
        "rank_p95": rounded(np.percentile(ranks, 95)),
        "correct_score_mean": rounded(correct.mean()),
        "correct_margin_mean": rounded(margins.mean()),
        "ranks": ranks,
        "correct_scores": correct,
        "margins": margins,
    }


def serializable_global_metrics(metrics: dict) -> dict:
    return {
        key: value
        for key, value in metrics.items()
        if key not in {"ranks", "correct_scores", "margins"}
    }


def aggregate_metric_dicts(metrics: list[dict]) -> dict:
    if not metrics:
        return {}
    output = {}
    for key in metrics[0]:
        values = [item[key] for item in metrics]
        if isinstance(values[0], (int, float)) and key != "count":
            output[key] = {
                "mean": rounded(np.mean(values)),
                "std": rounded(np.std(values)),
            }
        elif key == "count":
            output[key] = int(round(float(np.mean(values))))
    return output

## Calculation

In [ ]:
def result_output_path(experiment_mode: str) -> Path:
    if experiment_mode not in {"local", "global"}:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    stem = "training-free"
    if RECORD_LIMIT is not None:
        stem += f".limit-{int(RECORD_LIMIT)}"
    return OUTPUT_DIR / experiment_mode / embedding_precision_mode() / f"{stem}.json"


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    staging = path.with_suffix(path.suffix + ".staging")
    with staging.open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)
        file.write("\n")
    staging.replace(path)


def experiment_configuration(experiment_mode: str) -> dict:
    if experiment_mode == "local":
        settings = {
            "negative_count": LOCAL_NEGATIVE_COUNT,
            "query_limit": LOCAL_QUERY_LIMIT,
            "calibration_fraction": LOCAL_CALIBRATION_FRACTION,
            "random_seed": RANDOM_SEED,
            "cohorts": ["random", "challenge"],
        }
    elif experiment_mode == "global":
        settings = {
            "bank_sizes": GLOBAL_BANK_SIZES,
            "repeats": GLOBAL_REPEATS,
            "query_owner_count": GLOBAL_QUERY_OWNER_COUNT,
            "query_limit": GLOBAL_QUERY_LIMIT,
            "random_seed": RANDOM_SEED,
            "bank_sampling": "nested_prefixes_with_fixed_query_owners",
            "bank_universe": "retained_causality_owners",
        }
    else:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    return {
        "mode": experiment_mode,
        "record_limit": RECORD_LIMIT,
        "scorers": list(SCORER_ORDER),
        "settings": settings,
    }


def causality_dataset_metadata_matches(
    saved: dict | None,
    current: dict,
) -> bool:
    if saved is None:
        return False
    identity_fields = (
        "source",
        "rows",
        "queries",
        "prompt_fingerprint",
        "policy",
        "dataset_id",
        "repo",
        "config",
        "split",
    )
    return all(saved.get(field) == current.get(field) for field in identity_fields)


def result_matches_configuration(
    path: Path,
    experiment: dict,
    dataset_metadata: dict,
) -> bool:
    if not path.exists():
        return False
    with path.open(encoding="utf-8") as file:
        result = json.load(file)
    return (
        result.get("embedding_model") == EMBEDDING_MODEL_REPO
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("max_length") == EMBEDDING_MAX_LENGTH
        and result.get("experiment") == experiment
        and causality_dataset_metadata_matches(
            result.get("dataset"),
            dataset_metadata,
        )
    )


def scorer_arrays(
    query_embeddings: np.ndarray,
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict[str, np.ndarray]:
    prompt_scores = query_embeddings @ edit_prompt_embeddings.T
    subject_scores = query_embeddings @ subject_embeddings.T
    return {
        "bge_prompt": prompt_scores,
        "bge_subject_max": np.maximum(prompt_scores, subject_scores),
        "bge_composed": query_embeddings @ composed_embeddings.T,
    }


def local_diagnostics(
    observations: list[dict],
    records: list[dict],
    limit: int,
) -> list[dict]:
    failures = sorted(
        (
            item for item in observations
            if item["positive_score"] <= max(item["negative_scores"])
        ),
        key=lambda item: item["positive_score"] - max(item["negative_scores"]),
    )
    output = []
    for item in failures[:limit]:
        negatives = np.asarray(item["negative_scores"], dtype=np.float32)
        best_position = int(np.argmax(negatives))
        retrieved_index = item["negative_indices"][best_position]
        output.append({
            "query": item["query"]["prompt"],
            "source_edit_position": item["query"]["source_edit_position"],
            "expected_edit": item["query"]["expected_edit"],
            "retrieved_edit": records[retrieved_index]["edit"],
            "positive_score": rounded(item["positive_score"]),
            "retrieved_score": rounded(negatives[best_position]),
            "gap": rounded(item["positive_score"] - negatives[best_position]),
        })
    return output


def run_local_experiment(
    records: list[dict],
    queries: list[dict],
    bm25: Bm25Index,
    query_embedding_by_id: dict[str, np.ndarray],
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict:
    cohorts = build_local_cohorts(
        records,
        queries,
        bm25,
        LOCAL_NEGATIVE_COUNT,
        RANDOM_SEED,
        LOCAL_CALIBRATION_FRACTION,
    )
    observations = {
        method: {"random": [], "challenge": []}
        for method in SCORER_ORDER
    }
    for cohort in tqdm(cohorts, desc="Score local causal cohorts"):
        query = cohort["query"]
        candidates = cohort["candidate_indices"]
        query_embedding = query_embedding_by_id[query["query_id"]][None, :]
        embedding_scores = scorer_arrays(
            query_embedding,
            edit_prompt_embeddings[candidates],
            subject_embeddings[candidates],
            composed_embeddings[candidates],
        )
        method_scores = {
            "bm25_prompt": np.asarray(cohort["bm25_scores"], dtype=np.float32),
            **{key: value[0] for key, value in embedding_scores.items()},
        }
        for method, scores in method_scores.items():
            observations[method][cohort["cohort"]].append({
                "query": query,
                "positive_score": float(scores[0]),
                "negative_scores": [float(value) for value in scores[1:]],
                "negative_indices": list(candidates[1:]),
                "calibration": cohort["calibration"],
            })

    methods = {}
    for method in SCORER_ORDER:
        methods[method] = {"cohorts": {}}
        for cohort_name in ("random", "challenge"):
            current = observations[method][cohort_name]
            groups = {
                "all": current,
                "position_0": [
                    item for item in current
                    if item["query"]["source_edit_position"] == 0
                ],
                "position_1": [
                    item for item in current
                    if item["query"]["source_edit_position"] == 1
                ],
                "origin_single": [
                    item for item in current
                    if item["query"]["origin"] == "single"
                ],
                "origin_nested": [
                    item for item in current
                    if item["query"]["origin"] == "double.single_path1"
                ],
            }
            methods[method]["cohorts"][cohort_name] = {
                "metrics": {
                    group_name: local_observation_metrics(group)
                    for group_name, group in groups.items()
                },
                "threshold": threshold_metrics(current),
                "diagnostics": local_diagnostics(
                    [item for item in current if not item["calibration"]],
                    records,
                    DIAGNOSTIC_LIMIT,
                ),
            }
    cohort_payload = [
        {
            "query_id": item["query"]["query_id"],
            "cohort": item["cohort"],
            "candidate_indices": item["candidate_indices"],
            "calibration": item["calibration"],
        }
        for item in cohorts
    ]
    return {
        "query_count": len(queries),
        "cohort_fingerprint": cohort_fingerprint(cohort_payload),
        "methods": methods,
    }


def global_failure_examples(
    scores: np.ndarray,
    expected_positions: np.ndarray,
    queries: list[dict],
    bank_indices: np.ndarray,
    records: list[dict],
    limit: int,
) -> list[dict]:
    top_positions = np.argmax(scores, axis=1)
    failures = np.flatnonzero(top_positions != expected_positions)
    margins = scores[np.arange(len(scores)), expected_positions] - scores[
        np.arange(len(scores)), top_positions
    ]
    failures = failures[np.argsort(margins[failures])]
    output = []
    for query_position in failures[:limit]:
        retrieved_position = int(top_positions[query_position])
        output.append({
            "query": queries[int(query_position)]["prompt"],
            "source_edit_position": queries[int(query_position)]["source_edit_position"],
            "expected_edit": queries[int(query_position)]["expected_edit"],
            "retrieved_edit": records[int(bank_indices[retrieved_position])]["edit"],
            "expected_score": rounded(
                scores[int(query_position), int(expected_positions[query_position])]
            ),
            "retrieved_score": rounded(scores[int(query_position), retrieved_position]),
            "gap": rounded(margins[int(query_position)]),
        })
    return output


def run_global_experiment(
    records: list[dict],
    bm25: Bm25Index,
    cohorts: list[dict],
    query_embedding_by_id: dict[str, np.ndarray],
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict:
    scales = {}
    cohort_payload = []
    for cohort in tqdm(cohorts, desc="Run global causal retrieval"):
        queries = cohort["queries"]
        query_embeddings = np.stack([
            query_embedding_by_id[query["query_id"]]
            for query in queries
        ])
        bm25_full = np.stack([bm25.score(query["prompt"]) for query in queries])
        for requested_size, bank_size in cohort["bank_sizes"]:
            bank_indices = cohort["permutation"][:bank_size]
            position_by_owner = {
                int(owner_index): position
                for position, owner_index in enumerate(bank_indices)
            }
            expected_positions = np.asarray([
                position_by_owner[int(query["owner_index"])]
                for query in queries
            ], dtype=np.int64)
            embedding_scores = scorer_arrays(
                query_embeddings,
                edit_prompt_embeddings[bank_indices],
                subject_embeddings[bank_indices],
                composed_embeddings[bank_indices],
            )
            method_scores = {
                "bm25_prompt": bm25_full[:, bank_indices],
                **embedding_scores,
            }
            scale_key = "all" if requested_size is None else str(requested_size)
            scale = scales.setdefault(scale_key, {
                "requested_bank_size": requested_size,
                "bank_size": bank_size,
                "methods": {
                    method: {
                        "repeats": [],
                        "positions": {"0": [], "1": []},
                        "origins": {"single": [], "double.single_path1": []},
                    }
                    for method in SCORER_ORDER
                },
                "diagnostics": {},
            })
            for method, scores in method_scores.items():
                metrics = global_metrics(scores, expected_positions)
                scale["methods"][method]["repeats"].append(
                    serializable_global_metrics(metrics)
                )
                for position in (0, 1):
                    mask = np.asarray([
                        query["source_edit_position"] == position
                        for query in queries
                    ])
                    if mask.any():
                        position_metrics = global_metrics(
                            scores[mask], expected_positions[mask]
                        )
                        scale["methods"][method]["positions"][str(position)].append(
                            serializable_global_metrics(position_metrics)
                        )
                for origin in ("single", "double.single_path1"):
                    mask = np.asarray([
                        query["origin"] == origin
                        for query in queries
                    ])
                    if mask.any():
                        origin_metrics = global_metrics(
                            scores[mask], expected_positions[mask]
                        )
                        scale["methods"][method]["origins"][origin].append(
                            serializable_global_metrics(origin_metrics)
                        )
                if bank_size == cohort["bank_sizes"][-1][1]:
                    examples = global_failure_examples(
                        scores,
                        expected_positions,
                        queries,
                        bank_indices,
                        records,
                        DIAGNOSTIC_LIMIT,
                    )
                    scale["diagnostics"].setdefault(method, []).extend(examples[:1])
            cohort_payload.append({
                "repeat": cohort["repeat_index"],
                "requested_bank_size": requested_size,
                "bank_indices": [int(value) for value in bank_indices],
                "query_ids": [query["query_id"] for query in queries],
            })

    output_scales = []
    for scale in sorted(scales.values(), key=lambda item: item["bank_size"]):
        methods = {}
        for method in SCORER_ORDER:
            method_data = scale["methods"][method]
            methods[method] = {
                "aggregate": aggregate_metric_dicts(method_data["repeats"]),
                "positions": {
                    position: aggregate_metric_dicts(repeats)
                    for position, repeats in method_data["positions"].items()
                    if repeats
                },
                "origins": {
                    origin: aggregate_metric_dicts(repeats)
                    for origin, repeats in method_data["origins"].items()
                    if repeats
                },
                "repeats": method_data["repeats"],
            }
        output_scales.append({
            "requested_bank_size": scale["requested_bank_size"],
            "bank_size": scale["bank_size"],
            "methods": methods,
            "diagnostics": {
                method: examples[:DIAGNOSTIC_LIMIT]
                for method, examples in scale["diagnostics"].items()
            },
        })
    return {
        "cohort_fingerprint": cohort_fingerprint(cohort_payload),
        "scales": output_scales,
    }


def calculate_causality(
    experiment_mode: str,
    dataset_source: str,
    overwrite_output: bool,
) -> None:
    experiment = experiment_configuration(experiment_mode)
    dataset = load_causality_dataset(dataset_source)
    records = build_causality_records(dataset, RECORD_LIMIT)
    dataset_metadata = causality_dataset_metadata(dataset, records, dataset_source)
    output_path = result_output_path(experiment_mode)
    if output_path.exists() and not overwrite_output:
        if result_matches_configuration(output_path, experiment, dataset_metadata):
            print(f"skipped_existing={output_path}")
            return
        print(f"recalculating_stale={output_path}")

    all_queries = flatten_queries(records)
    if experiment_mode == "local":
        queries = sample_queries(
            all_queries,
            LOCAL_QUERY_LIMIT,
            RANDOM_SEED,
            "local-query-limit",
        )
        global_cohorts = None
        used_queries = queries
    else:
        queries = None
        global_cohorts = build_global_cohorts(
            records,
            GLOBAL_BANK_SIZES,
            GLOBAL_REPEATS,
            GLOBAL_QUERY_OWNER_COUNT,
            GLOBAL_QUERY_LIMIT,
            RANDOM_SEED,
        )
        used_query_by_id = {
            query["query_id"]: query
            for cohort in global_cohorts
            for query in cohort["queries"]
        }
        used_queries = list(used_query_by_id.values())
    edits = [record["edit"] for record in records]
    bm25 = Bm25Index([edit["prompt"] for edit in edits])
    model = load_embedding_model()
    try:
        edit_prompt_embeddings = encode_prompts(
            model, [edit["prompt"] for edit in edits]
        )
        subject_embeddings = encode_prompts(
            model, [edit["subject"] for edit in edits]
        )
        composed_embeddings = encode_prompts(
            model, [composed_edit_text(edit) for edit in edits]
        )
        query_embeddings = encode_prompts(
            model, [query["prompt"] for query in used_queries]
        )
        query_embedding_by_id = {
            query["query_id"]: query_embeddings[index]
            for index, query in enumerate(used_queries)
        }
        if experiment_mode == "local":
            analysis = run_local_experiment(
                records,
                queries,
                bm25,
                query_embedding_by_id,
                edit_prompt_embeddings,
                subject_embeddings,
                composed_embeddings,
            )
        else:
            analysis = run_global_experiment(
                records,
                bm25,
                global_cohorts,
                query_embedding_by_id,
                edit_prompt_embeddings,
                subject_embeddings,
                composed_embeddings,
            )
    finally:
        release_embedding_model(model)

    result = {
        "embedding_model": EMBEDDING_MODEL_REPO,
        "representation": "normalized_dense_sentence_embedding",
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length": EMBEDDING_MAX_LENGTH,
        "dataset": dataset_metadata,
        "experiment": experiment,
        "analysis": analysis,
    }
    write_json_atomic(output_path, result)
    print(f"saved={output_path}")

## Tables And Plots

In [ ]:
def load_result(experiment_mode: str) -> dict:
    path = result_output_path(experiment_mode)
    if not path.exists():
        raise FileNotFoundError(f"Calculate the experiment first: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def local_metric_table(result: dict) -> pd.DataFrame:
    rows = []
    for method in SCORER_ORDER:
        for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
            for group, metrics in cohort_data["metrics"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Negative cohort": cohort,
                    "Group": group,
                    "Queries": metrics.get("count", 0),
                    "AUC": metrics.get("pairwise_auc"),
                    "Top-1": metrics.get("top1_accuracy"),
                    "MRR": metrics.get("mrr"),
                    "Best-negative gap": metrics.get("mean_best_negative_gap"),
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Negative cohort", "Group"])


def local_threshold_table(result: dict) -> pd.DataFrame:
    rows = []
    for method in SCORER_ORDER:
        for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
            threshold = cohort_data["threshold"]
            rows.append({
                "Scorer": SCORER_LABELS[method],
                "Negative cohort": cohort,
                "Available": threshold.get("available", False),
                "Threshold": threshold.get("threshold"),
                "Positive recall": threshold.get("evaluation_positive_recall"),
                "Negative FPR": threshold.get("evaluation_negative_false_positive_rate"),
                "Precision": threshold.get("evaluation_precision"),
            })
    return pd.DataFrame(rows).set_index(["Scorer", "Negative cohort"])


def plot_local_metrics(result: dict) -> None:
    metrics = (
        ("pairwise_auc", "Pairwise AUC"),
        ("top1_accuracy", "Strict top-1"),
        ("mrr", "MRR"),
        ("mean_best_negative_gap", "Best-negative gap"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    x = np.arange(len(SCORER_ORDER))
    width = 0.36
    for axis, (metric_key, title) in zip(axes.flat, metrics):
        for offset, cohort in ((-width / 2, "random"), (width / 2, "challenge")):
            values = [
                result["analysis"]["methods"][method]["cohorts"][cohort]["metrics"]["all"][metric_key]
                for method in SCORER_ORDER
            ]
            axis.bar(x + offset, values, width=width, label=cohort)
        axis.set_title(title)
        axis.set_xticks(x, [SCORER_LABELS[method] for method in SCORER_ORDER], rotation=18, ha="right")
        axis.grid(axis="y", alpha=0.25)
        if metric_key == "mean_best_negative_gap":
            axis.axhline(0.0, color="black", linewidth=0.8)
    axes[0, 0].legend()
    fig.tight_layout()
    plt.show()


def global_metric_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            metrics = scale["methods"][method]["aggregate"]
            rows.append({
                "Scorer": SCORER_LABELS[method],
                "Bank size": scale["bank_size"],
                "R@1": metrics["recall_at_1"]["mean"],
                "R@5": metrics["recall_at_5"]["mean"],
                "R@10": metrics["recall_at_10"]["mean"],
                "R@20": metrics["recall_at_20"]["mean"],
                "R@50": metrics["recall_at_50"]["mean"],
                "MRR": metrics["mrr"]["mean"],
                "Rank p50": metrics["rank_p50"]["mean"],
                "Rank p95": metrics["rank_p95"]["mean"],
            })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size"])


def global_position_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            for position, metrics in scale["methods"][method]["positions"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Bank size": scale["bank_size"],
                    "Source position": int(position),
                    "R@1": metrics["recall_at_1"]["mean"],
                    "R@10": metrics["recall_at_10"]["mean"],
                    "MRR": metrics["mrr"]["mean"],
                    "Rank p50": metrics["rank_p50"]["mean"],
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size", "Source position"])


def global_origin_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            for origin, metrics in scale["methods"][method]["origins"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Bank size": scale["bank_size"],
                    "Origin": origin,
                    "R@1": metrics["recall_at_1"]["mean"],
                    "R@10": metrics["recall_at_10"]["mean"],
                    "MRR": metrics["mrr"]["mean"],
                    "Rank p50": metrics["rank_p50"]["mean"],
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size", "Origin"])


def plot_global_metrics(result: dict) -> None:
    metrics = (
        ("recall_at_1", "Recall@1"),
        ("recall_at_10", "Recall@10"),
        ("mrr", "MRR"),
        ("rank_p50", "Median rank"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for axis, (metric_key, title) in zip(axes.flat, metrics):
        for method in SCORER_ORDER:
            sizes = [scale["bank_size"] for scale in result["analysis"]["scales"]]
            means = [
                scale["methods"][method]["aggregate"][metric_key]["mean"]
                for scale in result["analysis"]["scales"]
            ]
            axis.plot(sizes, means, marker="o", label=SCORER_LABELS[method])
        axis.set_xscale("log")
        axis.set_title(title)
        axis.set_xlabel("Edit bank size")
        axis.grid(alpha=0.25)
        if metric_key in {"recall_at_1", "recall_at_10", "mrr"}:
            axis.set_ylim(0.0, 1.0)
        if metric_key == "rank_p50":
            axis.set_ylabel("Lower is better")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def diagnostic_table(result: dict, experiment_mode: str) -> pd.DataFrame:
    rows = []
    if experiment_mode == "local":
        for method in SCORER_ORDER:
            for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
                for example in cohort_data["diagnostics"]:
                    rows.append({
                        "Scorer": SCORER_LABELS[method],
                        "Setting": cohort,
                        "Position": example["source_edit_position"],
                        "Query": example["query"],
                        "Expected edit": example["expected_edit"]["prompt"],
                        "Retrieved edit": example["retrieved_edit"]["prompt"],
                        "Gap": example["gap"],
                    })
    else:
        for scale in result["analysis"]["scales"]:
            for method, examples in scale.get("diagnostics", {}).items():
                for example in examples:
                    rows.append({
                        "Scorer": SCORER_LABELS[method],
                        "Setting": scale["bank_size"],
                        "Position": example["source_edit_position"],
                        "Query": example["query"],
                        "Expected edit": example["expected_edit"]["prompt"],
                        "Retrieved edit": example["retrieved_edit"]["prompt"],
                        "Gap": example["gap"],
                    })
    return pd.DataFrame(rows)


def visualize_causality(experiment_mode: str) -> None:
    result = load_result(experiment_mode)
    if result["experiment"] != experiment_configuration(experiment_mode):
        raise ValueError("Saved result does not match the current experiment configuration.")
    current_dataset = load_causality_dataset(DATASET_SOURCE)
    current_records = build_causality_records(current_dataset, RECORD_LIMIT)
    current_metadata = causality_dataset_metadata(
        current_dataset,
        current_records,
        DATASET_SOURCE,
    )
    if not causality_dataset_metadata_matches(
        result.get("dataset"),
        current_metadata,
    ):
        raise ValueError(
            "Saved causality result does not match the currently loaded dataset "
            "or extraction policy. Recalculate before visualizing."
        )
    print(
        f"dataset_rows={result['dataset']['rows']} "
        f"causal_queries={result['dataset']['queries']} "
        f"cohort_fingerprint={result['analysis']['cohort_fingerprint']}"
    )
    if experiment_mode == "local":
        display(local_metric_table(result))
        display(local_threshold_table(result))
        plot_local_metrics(result)
        print(
            "Interpretation: high local AUC with low global Recall@1 means the scorer "
            "recognizes related edits but cannot select one reliably under competition. "
            "The threshold table is association-only because this dataset contains no "
            "unrelated locality/no-edit prompts."
        )
    else:
        display(global_metric_table(result))
        display(global_position_table(result))
        display(global_origin_table(result))
        plot_global_metrics(result)
        print(
            "Interpretation: stable recall as the retained causality-owner bank grows "
            "supports retrieval-based routing within this selected benchmark. "
            "A position gap shows whether questions that hide the edit subject (position 1) "
            "need a different decomposition or reranking signal."
        )
    diagnostics = diagnostic_table(result, experiment_mode)
    if len(diagnostics):
        display(diagnostics)

## Retrieval Configuration And Actions

Choose local association or global bank retrieval. The following configuration cell is immediately followed by the commented calculation and visualization actions.

In [ ]:
# Select one-edit association analysis or edit-bank retrieval.
# Options: "local", "global".
EXPERIMENT_MODE = "global"

# Load the causality split from local Arrow or Hugging Face.
# Options: "local", "huggingface".
DATASET_SOURCE = "local"

# Embedding inference device, precision, token limit, and batch size.
EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = True
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64

# Limit source records only for smoke tests; None uses the complete split.
RECORD_LIMIT = None

# Recalculate the selected result even when its fingerprints still match.
OVERWRITE_EXISTING_RESULT = True

# Shared deterministic cohort seed and bounded saved failures.
RANDOM_SEED = 42
DIAGNOSTIC_LIMIT = 5

# Local mode: one positive plus this many random or challenge negatives.
LOCAL_NEGATIVE_COUNT = 31
LOCAL_QUERY_LIMIT = None
LOCAL_CALIBRATION_FRACTION = 0.5

# Global mode: nested banks add distractors from retained causality owners.
# None means all 17,215 retained owners, not the full merged UniEdit universe.
GLOBAL_BANK_SIZES = [
    100,
    1000,
    2000,
    5000,
    None,
]
GLOBAL_REPEATS = 5
GLOBAL_QUERY_OWNER_COUNT = 100
GLOBAL_QUERY_LIMIT = None

In [ ]:
# Calculate and save the selected local or global experiment:
# calculate_causality(
#     experiment_mode=EXPERIMENT_MODE,
#     dataset_source=DATASET_SOURCE,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
# )

# Display the matching tables, plots, interpretation, and failures:
# visualize_causality(EXPERIMENT_MODE)

## Part II: Causal Intervention And Context Diagnostics

This stage holds retrieval constant by using the known owner edit. The experiments answer different failure questions:

- **Oracle hidden-state intervention:** compare base generation, normal Mnemo routing, and forced use of the correct memory row at several strengths. If forced routing fails, improving retrieval cannot fix that example because the stored vector itself does not support the required reasoning.
- **Structured and natural context:** prepend the correct edit fact or an unrelated control fact before both direct edit prompts and causal questions. Improvement only with correct context indicates that the model reads and uses the premise; equal improvement from unrelated context indicates a formatting or generation confound.
- **Prerequisite generation and likelihood:** test whether the unchanged hop needed to complete the chain is already available to the base model. A retrieved premise cannot produce the final answer when the model cannot recover this prerequisite.
- **Likelihood-screened natural context:** rerun premise injection on cases where the prerequisite appears available. This separates failure to use the edit from failure of the underlying unchanged knowledge.

Interpret source position 0 and 1 separately. At position 0, directly replacing the final prompt state with a vector trained to emit the edit answer is expected to risk stopping at the intermediate connector. At position 1, that same answer is already terminal, so answer injection is a more suitable mechanism. These diagnostics evaluate mechanisms; none is part of benchmark generation unless explicitly integrated later.

### Intervention Implementation

The following cells reproduce deterministic cohorts, model helpers, scoring, contextual controls, prerequisite probes, atomic result writing, and the forced-owner Mnemo diagnostic.

In [ ]:
def load_records(path: Path) -> list[dict]:
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row.get('edit'), dict) or not isinstance(row.get('causality'), list):
                raise ValueError(f'Invalid causality row at line {line_number}')
            records.append(row)
    return records

def owner_key(edit: dict) -> tuple[str, str]:
    return (str(edit['subject']['id']), str(edit['relation']['id']))

def dependency_keys(query: dict) -> list[tuple[str, str]]:
    return [(str(item['subject_id']), str(item['relation_id'])) for item in query.get('dependency_edit_keys', [])]

def collect_queries(records: list[dict]) -> tuple[dict[tuple[str, str], int], list[dict]]:
    owners = {owner_key(row['edit']): index for index, row in enumerate(records)}
    queries = []
    for row_index, row in enumerate(records):
        for query_index, query in enumerate(row['causality']):
            keys = dependency_keys(query)
            # The default data policy is independent edits: one causal query has one owner.
            if len(keys) != 1 or keys[0] not in owners:
                continue
            queries.append({
                'row_index': row_index,
                'query_index': query_index,
                'query': query,
                'owner_key': keys[0],
                'owner_row_index': owners[keys[0]],
            })
    return owners, queries

In [ ]:
def stratified_sample(items: list[dict], per_stratum: int, seed: int) -> list[dict]:
    groups = defaultdict(list)
    for item in items:
        query = item['query']
        groups[(query['source_edit_position'], query['origin'])].append(item)
    rng = random.Random(seed)
    selected = []
    for group_key in sorted(groups):
        group = list(groups[group_key])
        rng.shuffle(group)
        selected.extend(group[:per_stratum])
    selected.sort(key=lambda item: (item['row_index'], item['query_index']))
    return selected

def prepare_selected_queries(query_items: list[dict], label: str) -> None:
    global selected_queries, required_owner_keys, bank_keys, bank_rows, bank_index, context_items
    selected_queries = list(query_items)
    required_owner_keys = sorted({item['owner_key'] for item in selected_queries})
    if len(required_owner_keys) > MAX_EDIT_BANK:
        raise ValueError(
            f'Selected queries require {len(required_owner_keys)} owners, exceeding MAX_EDIT_BANK={MAX_EDIT_BANK}. '
            'Lower QUERY_LIMIT_PER_STRATUM or increase the explicit bank cap.'
        )
    distractor_keys = sorted(set(owners) - set(required_owner_keys))
    random.Random(SEED + 1).shuffle(distractor_keys)
    bank_keys = required_owner_keys + distractor_keys[:MAX_EDIT_BANK - len(required_owner_keys)]
    random.Random(SEED + 2).shuffle(bank_keys)
    bank_rows = [records[owners[key]] for key in bank_keys]
    bank_index = {key: index for index, key in enumerate(bank_keys)}
    for item in selected_queries:
        item['owner_memory_index'] = bank_index[item['owner_key']]
    context_items = build_context_items()
    print(f'prepared cohort={label}, selected queries={len(selected_queries)}, edit bank={len(bank_rows)}')
    print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in selected_queries))
    print(f'context queries={len(context_items)}')
    print(Counter(item['query_type'] for item in context_items))

def prepare_experiment() -> None:
    global records, owners, queries
    records = load_records(DATA_PATH)
    owners, queries = collect_queries(records)
    selected = stratified_sample(queries, QUERY_LIMIT_PER_STRATUM, SEED)
    print(f'loaded edits={len(records)}, eligible queries={len(queries)}')
    print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in queries))
    prepare_selected_queries(selected, 'default')

In [ ]:
def build_mnemo() -> Mnemo:
    config = load_config(MNEMO_CONFIG_PATH)
    config.external_only = EXTERNAL_ONLY
    return Mnemo(config)

def clean_text(text: str) -> str:
    text = text.strip().casefold()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

def generate_texts(
    editor: Mnemo,
    prompts: list[str],
    *,
    forced_indices: list[int] | None = None,
    strength: float | None = None,
) -> list[str]:
    if not prompts:
        return []
    if forced_indices is not None and len(forced_indices) != len(prompts):
        raise ValueError('Forced indices must align with prompts')
    tokenizer = editor.tokenizer
    model = editor.model
    original_padding = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    predictions = []
    try:
        for start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            batch_indices = None if forced_indices is None else forced_indices[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True).to(editor.device)
            generation_inputs = {
                **inputs,
                'max_new_tokens': MAX_NEW_TOKENS,
                'repetition_penalty': REPETITION_PENALTY,
                'do_sample': False,
                'pad_token_id': tokenizer.pad_token_id,
                'eos_token_id': tokenizer.eos_token_id,
                'routing_prompts': batch,
            }
            patch_context = (
                nullcontext()
                if batch_indices is None
                else editor.force_patch(batch_indices, strength=strength)
            )
            with patch_context, torch.inference_mode():
                output_ids = model.generate(**generation_inputs)
            prompt_length = inputs['input_ids'].shape[1]
            predictions.extend(tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True))
    finally:
        tokenizer.padding_side = original_padding
    return [prediction.strip() for prediction in predictions]

def answer_match(prediction: str, answers: list[str]) -> tuple[bool, str | None]:
    value = clean_text(prediction)
    if not value:
        return False, None
    normalized = [clean_text(answer) for answer in answers if clean_text(answer)]
    for answer in normalized:
        if value == answer or value.startswith(answer + ' '):
            return True, 'prefix'
    for answer in normalized:
        if f' {answer} ' in f' {value} ':
            return True, 'containment'
    return False, None

In [ ]:
def apply_owner_edits(editor: Mnemo) -> None:
    for row in tqdm(bank_rows, desc='Applying owner edits'):
        edit = row['edit']
        editor.edit(EditRequest(
            subject=edit['subject']['str'],
            prompt=edit['prompt'],
            answer=edit['answers'][0],
        ))
    if editor.external_memory_size != len(bank_rows):
        raise RuntimeError('The Mnemo memory bank is not aligned with sampled owners')

def run_predictions(editor: Mnemo) -> tuple[list[str], list[str], dict[str, list[str]]]:
    query_prompts = [item['query']['prompt'] for item in selected_queries]
    base_predictions = generate_texts(editor, query_prompts)
    apply_owner_edits(editor)
    normal_predictions = generate_texts(editor, query_prompts)
    owner_indices = [item['owner_memory_index'] for item in selected_queries]
    oracle_predictions = {
        str(strength): generate_texts(
            editor,
            query_prompts,
            forced_indices=owner_indices,
            strength=strength,
        )
        for strength in ORACLE_STRENGTHS
    }
    return base_predictions, normal_predictions, oracle_predictions

In [ ]:
# Kept as a separate definition cell so the expensive run is explicit below.
def compare_scores(base_scores: list[dict], candidate_scores: list[dict]) -> dict:
    groups = defaultdict(lambda: {'count': 0, 'incorrect_to_correct': 0, 'correct_to_incorrect': 0, 'net_gain': 0})
    for item, base, candidate in zip(selected_queries, base_scores, candidate_scores, strict=True):
        query = item['query']
        keys = ('overall', f'position_{query["source_edit_position"]}')
        for key in keys:
            groups[key]['count'] += 1
            if not base['matched'] and candidate['matched']:
                groups[key]['incorrect_to_correct'] += 1
            if base['matched'] and not candidate['matched']:
                groups[key]['correct_to_incorrect'] += 1
    for values in groups.values():
        values['net_gain'] = values['incorrect_to_correct'] - values['correct_to_incorrect']
    return dict(sorted(groups.items()))

In [ ]:
def score_selected_predictions(
    base_predictions: list[str],
    normal_predictions: list[str],
    oracle_predictions: dict[str, list[str]],
) -> tuple[list[dict], list[dict], dict[str, list[dict]]]:
    base_scores = score_condition(base_predictions)
    normal_scores = score_condition(normal_predictions)
    oracle_scores = {strength: score_condition(predictions) for strength, predictions in oracle_predictions.items()}
    return base_scores, normal_scores, oracle_scores

In [ ]:
def score_condition(predictions: list[str]) -> list[dict]:
    scored = []
    for item, prediction in zip(selected_queries, predictions, strict=True):
        matched, mode = answer_match(prediction, item['query']['answers'])
        scored.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
    return scored

def summarize(scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(selected_queries, scores, strict=True):
        query = item['query']
        groups['overall'].append(score['matched'])
        groups[f'position_{query["source_edit_position"]}'].append(score['matched'])
        groups[f'{query["origin"]}:position_{query["source_edit_position"]}'].append(score['matched'])
    return {key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None} for key, values in sorted(groups.items())}

In [ ]:
def build_context_items() -> list[dict]:
    if len(bank_rows) < 2:
        raise ValueError('Context intervention requires at least two bank rows')
    items = []
    for sample_index, selected in enumerate(selected_queries):
        edit = records[selected['owner_row_index']]['edit']
        common = {
            'sample_index': sample_index,
            'row_index': selected['row_index'],
            'query_index': selected['query_index'],
            'owner_key': selected['owner_key'],
            'owner_memory_index': selected['owner_memory_index'],
            'wrong_memory_index': (selected['owner_memory_index'] + 1) % len(bank_rows),
            'edit': edit,
        }
        items.append({
            **common,
            'query_type': 'simple',
            'prompt': edit['prompt'],
            'answers': edit['answers'],
            'source_edit_position': None,
            'origin': None,
        })
        query = selected['query']
        items.append({
            **common,
            'query_type': 'causal',
            'prompt': query['prompt'],
            'answers': query['answers'],
            'source_edit_position': query['source_edit_position'],
            'origin': query['origin'],
        })
    return items

def render_context(edit: dict, query: str, template: str) -> str:
    edit_prompt = edit['prompt'].strip()
    edit_answer = edit['answers'][0].strip()
    if template == 'new_fact':
        return f'New Fact: {edit_prompt} {edit_answer}\nPrompt: {query}'
    if template == 'qa':
        return (
            f'Updated knowledge:\nPrompt: {edit_prompt}\nAnswer: {edit_answer}'
            f'\n\nPrompt: {query}\nAnswer:'
        )
    if template == 'natural':
        if edit_prompt.endswith('?'):
            premise = f'that the answer to "{edit_prompt}" is "{edit_answer}"'
        else:
            premise = f'the new fact that {edit_prompt} {edit_answer}'
        return f'Given {premise}, {query}'
    raise ValueError(f'Unsupported context template: {template!r}')

def build_context_model():
    config = load_config(MNEMO_CONFIG_PATH)
    model_kwargs = {'dtype': torch.bfloat16 if config.fp16 else torch.float32}
    if config.model_parallel:
        model_kwargs['device_map'] = 'auto'
    model = AutoModelForCausalLM.from_pretrained(config.model_name, **model_kwargs)
    if not config.model_parallel:
        model = model.to(config.device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    device = get_model_device(model, fallback=config.device)
    return model, tokenizer, device

In [ ]:
def generate_context_texts(model, tokenizer, device, prompts: list[str]) -> list[str]:
    original_padding = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    predictions = []
    try:
        for start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True).to(device)
            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    repetition_penalty=REPETITION_PENALTY,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            prompt_length = inputs['input_ids'].shape[1]
            predictions.extend(
                tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True)
            )
    finally:
        tokenizer.padding_side = original_padding
    return [prediction.strip() for prediction in predictions]

def build_context_prompts(condition: str) -> list[str]:
    if condition == 'base':
        return [item['prompt'] for item in context_items]
    context_kind, template = condition.split(':', maxsplit=1)
    prompts = []
    for item in context_items:
        if context_kind == 'correct':
            edit = item['edit']
        elif context_kind == 'wrong':
            edit = bank_rows[item['wrong_memory_index']]['edit']
        else:
            raise ValueError(f'Unsupported context condition: {condition!r}')
        prompts.append(render_context(edit, item['prompt'], template))
    return prompts

def context_conditions() -> list[str]:
    conditions = ['base']
    conditions.extend(f'correct:{template}' for template in CONTEXT_TEMPLATES)
    if INCLUDE_WRONG_CONTEXT:
        conditions.extend(f'wrong:{template}' for template in CONTEXT_TEMPLATES)
    return conditions

In [ ]:
def score_context_condition(predictions: list[str]) -> list[dict]:
    scores = []
    for item, prediction in zip(context_items, predictions, strict=True):
        matched, mode = answer_match(prediction, item['answers'])
        scores.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
    return scores

def context_group_keys(item: dict) -> tuple[str, ...]:
    keys = ['overall', item['query_type']]
    if item['query_type'] == 'causal':
        keys.append(f'causal:position_{item["source_edit_position"]}')
        keys.append(f'{item["origin"]}:position_{item["source_edit_position"]}')
    return tuple(keys)

def summarize_context(scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(context_items, scores, strict=True):
        for key in context_group_keys(item):
            groups[key].append(score['matched'])
    return {
        key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None}
        for key, values in sorted(groups.items())
    }

def compare_context_scores(base_scores: list[dict], candidate_scores: list[dict]) -> dict:
    groups = defaultdict(lambda: {
        'count': 0,
        'incorrect_to_correct': 0,
        'correct_to_incorrect': 0,
        'net_gain': 0,
    })
    for item, base, candidate in zip(context_items, base_scores, candidate_scores, strict=True):
        for key in context_group_keys(item):
            groups[key]['count'] += 1
            if not base['matched'] and candidate['matched']:
                groups[key]['incorrect_to_correct'] += 1
            if base['matched'] and not candidate['matched']:
                groups[key]['correct_to_incorrect'] += 1
    for values in groups.values():
        values['net_gain'] = values['incorrect_to_correct'] - values['correct_to_incorrect']
    return dict(sorted(groups.items()))

In [ ]:
def context_fingerprint() -> str:
    payload = '\n'.join(
        f'{item["query_type"]}\t{item["prompt"]}' for item in context_items
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def build_context_payload(
    prompts: dict[str, list[str]],
    predictions: dict[str, list[str]],
    scores: dict[str, list[dict]],
) -> dict:
    rows = []
    for index, item in enumerate(context_items):
        rows.append({
            'sample_index': item['sample_index'],
            'row_index': item['row_index'],
            'query_index': item['query_index'],
            'query_type': item['query_type'],
            'prompt': item['prompt'],
            'answers': item['answers'],
            'source_edit_position': item['source_edit_position'],
            'origin': item['origin'],
            'owner_key': list(item['owner_key']),
            'owner_memory_index': item['owner_memory_index'],
            'wrong_memory_index': item['wrong_memory_index'],
            'edit': {
                'subject': item['edit']['subject']['str'],
                'prompt': item['edit']['prompt'],
                'answer': item['edit']['answers'][0],
            },
            'conditions': {
                condition: {
                    'input': prompts[condition][index],
                    **scores[condition][index],
                }
                for condition in prompts
            },
        })
    base_scores = scores['base']
    return {
        'schema_version': 1,
        'experiment': 'contextual_premise_oracle',
        'model_key': MODEL_KEY,
        'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
        'seed': SEED,
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'edit_bank_size': len(bank_rows),
        'context_templates': list(CONTEXT_TEMPLATES),
        'include_wrong_context': INCLUDE_WRONG_CONTEXT,
        'dataset_fingerprint': context_fingerprint(),
        'summary': {condition: summarize_context(value) for condition, value in scores.items()},
        'comparisons_vs_base': {
            condition: compare_context_scores(base_scores, value)
            for condition, value in scores.items()
            if condition != 'base'
        },
        'rows': rows,
    }

def run_context_experiment() -> None:
    prepare_experiment()
    model, tokenizer, device = build_context_model()
    try:
        prompts = {condition: build_context_prompts(condition) for condition in context_conditions()}
        predictions = {}
        for condition, condition_prompts in prompts.items():
            print(f'generating condition={condition} records={len(condition_prompts)}')
            predictions[condition] = generate_context_texts(
                model, tokenizer, device, condition_prompts
            )
        scores = {
            condition: score_context_condition(value)
            for condition, value in predictions.items()
        }
        payload = build_context_payload(prompts, predictions, scores)
        atomic_save(payload, CONTEXT_OUTPUT_PATH)
        for condition, summary in payload['summary'].items():
            print(condition, summary)
        print(f'saved {CONTEXT_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_natural_context_experiment() -> None:
    conditions = ('base', 'correct:natural', 'wrong:natural')
    prepare_experiment()
    model, tokenizer, device = build_context_model()
    try:
        prompts = {condition: build_context_prompts(condition) for condition in conditions}
        predictions = {}
        for condition, condition_prompts in prompts.items():
            print(f'generating condition={condition} records={len(condition_prompts)}')
            predictions[condition] = generate_context_texts(
                model, tokenizer, device, condition_prompts
            )
        scores = {
            condition: score_context_condition(value)
            for condition, value in predictions.items()
        }
        payload = build_context_payload(prompts, predictions, scores)
        payload['experiment'] = 'natural_contextual_premise_oracle'
        payload['context_templates'] = ['natural']
        atomic_save(payload, NATURAL_OUTPUT_PATH)
        for condition, summary in payload['summary'].items():
            print(condition, summary)
        print(f'saved {NATURAL_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def build_prerequisite_items(query_items: list[dict]) -> list[dict]:
    items = []
    for sample_index, selected in enumerate(query_items):
        query = selected['query']
        hops = query.get('hops')
        if not isinstance(hops, list) or len(hops) != 2:
            raise ValueError(f'Expected exactly two hops for sample {sample_index}')
        source_position = query['source_edit_position']
        if source_position not in (0, 1):
            raise ValueError(f'Invalid source edit position for sample {sample_index}: {source_position!r}')
        if not hops[source_position].get('is_source_edit'):
            raise ValueError(f'Source hop is not marked as the edit for sample {sample_index}')
        prerequisite_index = 1 - source_position
        prerequisite = hops[prerequisite_index]
        if prerequisite.get('is_source_edit'):
            raise ValueError(f'Prerequisite hop is marked as the source edit for sample {sample_index}')
        items.append({
            'sample_index': sample_index,
            'row_index': selected['row_index'],
            'query_index': selected['query_index'],
            'causal_prompt': query['prompt'],
            'source_edit_position': source_position,
            'origin': query['origin'],
            'owner_key': selected['owner_key'],
            'prerequisite_hop_index': prerequisite_index,
            'prompt': prerequisite['prompt'],
            'answers': prerequisite['answers'],
        })
    return items

def summarize_prerequisites(items: list[dict], scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(items, scores, strict=True):
        position = item['source_edit_position']
        groups['overall'].append(score)
        groups[f'position_{position}'].append(score)
        groups[f'{item["origin"]}:position_{position}'].append(score)
    return {
        key: {
            'count': len(values),
            'accuracy': sum(score['matched'] for score in values) / len(values) if values else None,
            'strict_prefix_accuracy': (
                sum(score['match_mode'] == 'prefix' for score in values) / len(values)
                if values else None
            ),
        }
        for key, values in sorted(groups.items())
    }

def run_prerequisite_experiment() -> None:
    prepare_experiment()
    prerequisite_items = build_prerequisite_items(selected_queries)
    model, tokenizer, device = build_context_model()
    try:
        prompts = [item['prompt'] for item in prerequisite_items]
        print(f'generating prerequisite records={len(prompts)}')
        predictions = generate_context_texts(model, tokenizer, device, prompts)
        scores = []
        for item, prediction in zip(prerequisite_items, predictions, strict=True):
            matched, mode = answer_match(prediction, item['answers'])
            scores.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
        payload = {
            'schema_version': 1,
            'experiment': 'prerequisite_hop_control',
            'model_key': MODEL_KEY,
            'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
            'seed': SEED,
            'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
            'causal_dataset_fingerprint': fingerprint(selected_queries),
            'summary': summarize_prerequisites(prerequisite_items, scores),
            'rows': [
                {
                    **item,
                    'owner_key': list(item['owner_key']),
                    **score,
                }
                for item, score in zip(prerequisite_items, scores, strict=True)
            ],
        }
        atomic_save(payload, PREREQUISITE_OUTPUT_PATH)
        print(payload['summary'])
        print(f'saved {PREREQUISITE_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_prerequisite_screening() -> None:
    screen_records = load_records(DATA_PATH)
    _, eligible_queries = collect_queries(screen_records)
    screen_queries = stratified_sample(
        eligible_queries, PREREQUISITE_SCREEN_LIMIT_PER_STRATUM, SEED
    )
    prerequisite_items = build_prerequisite_items(screen_queries)
    print(f'eligible causal queries={len(eligible_queries)}')
    print(Counter((item['source_edit_position'], item['origin']) for item in prerequisite_items))
    model, tokenizer, device = build_context_model()
    try:
        prompts = [item['prompt'] for item in prerequisite_items]
        print(f'generating prerequisite screening records={len(prompts)}')
        predictions = generate_context_texts(model, tokenizer, device, prompts)
        scores = []
        for item, prediction in zip(prerequisite_items, predictions, strict=True):
            matched, mode = answer_match(prediction, item['answers'])
            scores.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
        payload = {
            'schema_version': 1,
            'experiment': 'prerequisite_hop_screening',
            'model_key': MODEL_KEY,
            'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
            'seed': SEED,
            'query_limit_per_stratum': PREREQUISITE_SCREEN_LIMIT_PER_STRATUM,
            'causal_dataset_fingerprint': fingerprint(screen_queries),
            'summary': summarize_prerequisites(prerequisite_items, scores),
            'rows': [
                {
                    **item,
                    'owner_key': list(item['owner_key']),
                    **score,
                }
                for item, score in zip(prerequisite_items, scores, strict=True)
            ],
        }
        atomic_save(payload, PREREQUISITE_SCREEN_OUTPUT_PATH)
        print(payload['summary'])
        print(f'saved {PREREQUISITE_SCREEN_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
def prepare_likelihood_screened_experiment() -> None:
    global records, owners, queries, likelihood_selection
    likelihood_payload = json.loads(
        PREREQUISITE_LIKELIHOOD_OUTPUT_PATH.read_text(encoding='utf-8')
    )
    records = load_records(DATA_PATH)
    owners, queries = collect_queries(records)
    query_lookup = {(item['row_index'], item['query_index']): item for item in queries}
    groups = defaultdict(list)
    for row in likelihood_payload['rows']:
        if row['length_normalized_rank'] == 1 and row['sequence_rank'] == 1:
            groups[(row['source_edit_position'], row['origin'])].append(row)
    selected_rows = []
    available = {}
    for group_key in sorted(groups):
        candidates = list(groups[group_key])
        available[f'{group_key[0]}:{group_key[1]}'] = len(candidates)
        if len(candidates) < QUERY_LIMIT_PER_STRATUM:
            raise ValueError(
                f'Likelihood-screened stratum {group_key} has {len(candidates)} rows; '
                f'need {QUERY_LIMIT_PER_STRATUM}'
            )
        random.Random(f'{SEED + 10}:{group_key[0]}:{group_key[1]}').shuffle(candidates)
        selected_rows.extend(candidates[:QUERY_LIMIT_PER_STRATUM])
    selected_rows.sort(key=lambda row: (row['row_index'], row['query_index']))
    selected = []
    for row in selected_rows:
        key = (row['row_index'], row['query_index'])
        if key not in query_lookup:
            raise ValueError(f'Likelihood row is not an eligible query: {key}')
        selected.append(query_lookup[key])
    likelihood_selection = {
        'source_path': str(PREREQUISITE_LIKELIHOOD_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
        'criterion': 'length_normalized_rank == 1 and sequence_rank == 1',
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'available_by_stratum': available,
        'selected_count': len(selected),
        'source_dataset_fingerprint': likelihood_payload['dataset_fingerprint'],
        'selected_likelihood_sample_indices': [row['sample_index'] for row in selected_rows],
    }
    prepare_selected_queries(selected, 'likelihood-screened')

def run_likelihood_screened_natural_experiment() -> None:
    conditions = ('base', 'correct:natural', 'wrong:natural')
    prepare_likelihood_screened_experiment()
    model, tokenizer, device = build_context_model()
    try:
        prompts = {condition: build_context_prompts(condition) for condition in conditions}
        predictions = {}
        for condition, condition_prompts in prompts.items():
            print(f'generating condition={condition} records={len(condition_prompts)}')
            predictions[condition] = generate_context_texts(
                model, tokenizer, device, condition_prompts
            )
        scores = {
            condition: score_context_condition(value)
            for condition, value in predictions.items()
        }
        payload = build_context_payload(prompts, predictions, scores)
        payload['experiment'] = 'likelihood_screened_natural_context'
        payload['context_templates'] = ['natural']
        payload['likelihood_selection'] = likelihood_selection
        payload['causal_dataset_fingerprint'] = fingerprint(selected_queries)
        atomic_save(payload, NATURAL_LIKELIHOOD_OUTPUT_PATH)
        for condition, summary in payload['summary'].items():
            print(condition, summary)
        print(f'saved {NATURAL_LIKELIHOOD_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
def clean_answer_set(answers: list[str]) -> frozenset[str]:
    return frozenset(clean_text(answer) for answer in answers if clean_text(answer))

def build_prerequisite_likelihood_items() -> tuple[list[dict], dict]:
    screen_payload = json.loads(PREREQUISITE_SCREEN_OUTPUT_PATH.read_text(encoding='utf-8'))
    screen_rows = screen_payload['rows']
    screen_records = load_records(DATA_PATH)
    pools = defaultdict(dict)
    enriched = []
    for row in screen_rows:
        query = screen_records[row['row_index']]['causality'][row['query_index']]
        hop = query['hops'][row['prerequisite_hop_index']]
        if hop['prompt'] != row['prompt'] or hop['answers'] != row['answers']:
            raise ValueError(f'Screen row does not match source data at sample {row["sample_index"]}')
        relation_key = (str(hop['relation']['id']), str(hop.get('direction', 'forward')))
        answer_set = clean_answer_set(hop['answers'])
        primary = hop['answers'][0].strip()
        enriched_item = {
            'screen_row': row,
            'query': query,
            'hop': hop,
            'relation_key': relation_key,
            'answer_set': answer_set,
            'primary_answer': primary,
        }
        enriched.append(enriched_item)
        pools[relation_key].setdefault(clean_text(primary), {
            'answer': primary,
            'answer_set': answer_set,
        })
    items = []
    skipped = Counter()
    for item in enriched:
        distractors = []
        for candidate in pools[item['relation_key']].values():
            if not candidate['answer_set'] or candidate['answer_set'] & item['answer_set']:
                continue
            distractors.append(candidate['answer'])
        if len(distractors) < LIKELIHOOD_DISTRACTOR_COUNT:
            row = item['screen_row']
            skipped[(row['source_edit_position'], row['origin'])] += 1
            continue
        rng = random.Random(
            f'{SEED}:{item["screen_row"]["row_index"]}:{item["screen_row"]["query_index"]}'
        )
        rng.shuffle(distractors)
        candidates = [item['primary_answer'], *distractors[:LIKELIHOOD_DISTRACTOR_COUNT]]
        rng.shuffle(candidates)
        items.append({
            'sample_index': item['screen_row']['sample_index'],
            'row_index': item['screen_row']['row_index'],
            'query_index': item['screen_row']['query_index'],
            'source_edit_position': item['screen_row']['source_edit_position'],
            'origin': item['screen_row']['origin'],
            'relation_id': item['relation_key'][0],
            'direction': item['relation_key'][1],
            'prompt': item['hop']['prompt'],
            'answers': item['hop']['answers'],
            'candidates': candidates,
            'correct_answer': item['primary_answer'],
        })
    return items, {
        'screen_rows': len(screen_rows),
        'skipped': sum(skipped.values()),
        'skipped_by_stratum': {
            f'{position}:{origin}': count
            for (position, origin), count in sorted(skipped.items())
        },
    }

def score_answer_candidates(model, tokenizer, device, pairs: list[tuple[str, str]]) -> list[dict]:
    if not pairs:
        return []
    scores = []
    pad_id = tokenizer.pad_token_id
    for start in range(0, len(pairs), LIKELIHOOD_BATCH_SIZE):
        batch = pairs[start:start + LIKELIHOOD_BATCH_SIZE]
        sequences = []
        answer_starts = []
        for prompt, answer in batch:
            prompt_ids = tokenizer.encode(prompt.rstrip(), add_special_tokens=False)
            answer_ids = tokenizer.encode(' ' + answer.strip(), add_special_tokens=False)
            if not prompt_ids or not answer_ids:
                raise ValueError('Likelihood probe received an empty prompt or answer')
            sequences.append(prompt_ids + answer_ids)
            answer_starts.append(len(prompt_ids))
        max_length = max(len(sequence) for sequence in sequences)
        input_ids = torch.full((len(sequences), max_length), pad_id, dtype=torch.long, device=device)
        attention_mask = torch.zeros_like(input_ids)
        answer_masks = torch.zeros((len(sequences), max_length - 1), dtype=torch.bool, device=device)
        for index, (sequence, answer_start) in enumerate(zip(sequences, answer_starts, strict=True)):
            length = len(sequence)
            input_ids[index, :length] = torch.tensor(sequence, dtype=torch.long, device=device)
            attention_mask[index, :length] = 1
            answer_masks[index, answer_start - 1:length - 1] = True
        with torch.inference_mode():
            logits = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits
            token_nll = F.cross_entropy(
                logits[:, :-1, :].float().transpose(1, 2),
                input_ids[:, 1:],
                reduction='none',
            )
        for index in range(len(batch)):
            answer_log_probabilities = -token_nll[index][answer_masks[index]]
            scores.append({
                'token_count': answer_log_probabilities.numel(),
                'mean_log_probability': answer_log_probabilities.mean().item(),
                'total_log_probability': answer_log_probabilities.sum().item(),
            })
        del logits, token_nll, input_ids, attention_mask, answer_masks
    return scores

def summarize_likelihood(items: list[dict]) -> dict:
    groups = defaultdict(list)
    for item in items:
        position = item['source_edit_position']
        keys = ('overall', f'position_{position}', f'{item["origin"]}:position_{position}')
        for key in keys:
            groups[key].append(item)
    summary = {}
    for key, values in sorted(groups.items()):
        normalized_ranks = [item['length_normalized_rank'] for item in values]
        sequence_ranks = [item['sequence_rank'] for item in values]
        summary[key] = {
            'count': len(values),
            'length_normalized_top1_accuracy': (
                sum(rank == 1 for rank in normalized_ranks) / len(normalized_ranks)
            ),
            'length_normalized_mrr': sum(1 / rank for rank in normalized_ranks) / len(normalized_ranks),
            'length_normalized_mean_margin': (
                sum(item['length_normalized_margin'] for item in values) / len(values)
            ),
            'sequence_top1_accuracy': sum(rank == 1 for rank in sequence_ranks) / len(sequence_ranks),
            'sequence_mrr': sum(1 / rank for rank in sequence_ranks) / len(sequence_ranks),
            'sequence_mean_margin': sum(item['sequence_margin'] for item in values) / len(values),
        }
    return summary

def run_prerequisite_likelihood_experiment() -> None:
    items, coverage = build_prerequisite_likelihood_items()
    model, tokenizer, device = build_context_model()
    try:
        pairs = [(item['prompt'], candidate) for item in items for candidate in item['candidates']]
        print(f'likelihood candidates={len(pairs)} from prerequisite rows={len(items)}')
        scores = score_answer_candidates(model, tokenizer, device, pairs)
        rows = []
        for item, offset in zip(items, range(0, len(scores), 1 + LIKELIHOOD_DISTRACTOR_COUNT), strict=True):
            candidate_scores = scores[offset:offset + 1 + LIKELIHOOD_DISTRACTOR_COUNT]
            correct_index = item['candidates'].index(item['correct_answer'])
            normalized_scores = [score['mean_log_probability'] for score in candidate_scores]
            sequence_scores = [score['total_log_probability'] for score in candidate_scores]
            correct_normalized = normalized_scores[correct_index]
            correct_sequence = sequence_scores[correct_index]
            rows.append({
                **item,
                'candidates': [
                    {
                        'answer': answer,
                        'is_correct': index == correct_index,
                        **candidate_scores[index],
                    }
                    for index, answer in enumerate(item['candidates'])
                ],
                'length_normalized_rank': 1 + sum(
                    score > correct_normalized for score in normalized_scores
                ),
                'length_normalized_margin': correct_normalized - max(
                    score for index, score in enumerate(normalized_scores) if index != correct_index
                ),
                'sequence_rank': 1 + sum(score > correct_sequence for score in sequence_scores),
                'sequence_margin': correct_sequence - max(
                    score for index, score in enumerate(sequence_scores) if index != correct_index
                ),
            })
        payload = {
            'schema_version': 1,
            'experiment': 'prerequisite_relation_likelihood',
            'model_key': MODEL_KEY,
            'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
            'screen_path': str(PREREQUISITE_SCREEN_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
            'seed': SEED,
            'distractor_count': LIKELIHOOD_DISTRACTOR_COUNT,
            'dataset_fingerprint': hashlib.sha256(
                '\n'.join(item['prompt'] for item in items).encode('utf-8')
            ).hexdigest(),
            'coverage': coverage,
            'summary': summarize_likelihood(rows),
            'rows': rows,
        }
        atomic_save(payload, PREREQUISITE_LIKELIHOOD_OUTPUT_PATH)
        print(payload['summary'])
        print(f'saved {PREREQUISITE_LIKELIHOOD_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
def fingerprint(items: list[dict]) -> str:
    payload = '\n'.join(item['query']['prompt'] for item in items).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def atomic_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=True, indent=2), encoding='utf-8')
    temporary.replace(path)

def build_payload(base_predictions, normal_predictions, oracle_predictions, base_scores, normal_scores, oracle_scores) -> dict:
    result_rows = []
    for index, item in enumerate(selected_queries):
        query = item['query']
        result_rows.append({
            'query_index': index,
            'prompt': query['prompt'],
            'answers': query['answers'],
            'source_edit_position': query['source_edit_position'],
            'origin': query['origin'],
            'owner_key': list(item['owner_key']),
            'owner_memory_index': item['owner_memory_index'],
            'base_prediction': base_predictions[index],
            'normal_prediction': normal_predictions[index],
            'base_match': base_scores[index],
            'normal_match': normal_scores[index],
            'oracle': {
                strength: {**oracle_scores[strength][index], 'prediction': oracle_predictions[strength][index]}
                for strength in oracle_predictions
            },
        })
    return {
        'schema_version': 1,
        'experiment': 'causal_intervention_oracle',
        'model_key': MODEL_KEY,
        'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
        'seed': SEED,
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'edit_bank_size': len(bank_rows),
        'oracle_strengths': list(ORACLE_STRENGTHS),
        'dataset_fingerprint': fingerprint(selected_queries),
        'summary': {
            'base': summarize(base_scores),
            'normal': summarize(normal_scores),
            'oracle': {strength: summarize(scores) for strength, scores in oracle_scores.items()},
        },
        'comparisons': {
            'normal_vs_base': compare_scores(base_scores, normal_scores),
            'oracle_vs_base': {strength: compare_scores(base_scores, scores) for strength, scores in oracle_scores.items()},
        },
        'edit_bank': [
            {
                'memory_index': index,
                'owner_key': list(key),
                'subject': row['edit']['subject']['str'],
                'prompt': row['edit']['prompt'],
                'answer': row['edit']['answers'][0],
            }
            for index, (key, row) in enumerate(zip(bank_keys, bank_rows, strict=True))
        ],
        'rows': result_rows,
    }

def run_oracle_experiment() -> None:
    prepare_experiment()
    editor = build_mnemo()
    try:
        base_predictions, normal_predictions, oracle_predictions = run_predictions(editor)
        base_scores, normal_scores, oracle_scores = score_selected_predictions(
            base_predictions, normal_predictions, oracle_predictions
        )
        payload = build_payload(
            base_predictions, normal_predictions, oracle_predictions,
            base_scores, normal_scores, oracle_scores,
        )
        atomic_save(payload, ORACLE_OUTPUT_PATH)
        print('base', payload['summary']['base'])
        print('normal', payload['summary']['normal'])
        for strength, summary in payload['summary']['oracle'].items():
            print(f'oracle strength={strength}', summary)
        print(f'saved {ORACLE_OUTPUT_PATH}')
    finally:
        editor.reset()

### Established Result Interpretation

The completed Qwen2.5-7B retrieval result shows that composed `subject + edit prompt + answer` embeddings are strong when the edited fact is visible in the causal question, but much weaker when the owner is hidden behind the first hop. This means semantic retrieval can be a useful candidate generator, especially for a capped bank, but it is not a complete causal router.

The forced-owner hidden-state result separates that retrieval limitation from intervention failure: full replacement answered `0/16` source-position-0 questions and `8/16` source-position-1 questions. Most position-0 failures emitted the intermediate edited answer, demonstrating that correct retrieval plus answer-state replacement does not perform two-hop propagation.

Natural premise experiments show a smaller but real relevance signal after prerequisite screening: correct context improves causal accuracy while unrelated context hurts it. The absolute accuracy remains too low for production integration. The current verdict is therefore: semantic retrieval is useful, causal premise use is possible, but neither hidden-state answer injection nor simple textual prepending yet supplies a reliable general causal mechanism.

## Intervention Configuration And Actions

The configuration below controls only the model-level diagnostics and is immediately followed by the commented actions in experimental order.

In [ ]:
# Experiment controls. Increase limits only after the smoke run looks sensible.
MODEL_KEY = 'qwen2.5-7b'
MNEMO_CONFIG_PATH = PROJECT_ROOT / 'src/config/mnemo/qwen2.5-7b.yaml'
DATA_PATH = PROJECT_ROOT / 'data/datasets/uniedit/clean/causality.jsonl'
ORACLE_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'oracle.json'
CONTEXT_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'context.json'
NATURAL_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'natural.json'
PREREQUISITE_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'prerequisite.24.json'
PREREQUISITE_SCREEN_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'prerequisite_screen.500.json'
PREREQUISITE_LIKELIHOOD_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'prerequisite_likelihood.500.json'
NATURAL_LIKELIHOOD_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'natural.likelihood.24.json'
SEED = 0
QUERY_LIMIT_PER_STRATUM = 24
# The broad screen samples up to this many queries per source-position/origin stratum.
PREREQUISITE_SCREEN_LIMIT_PER_STRATUM = 500
# The likelihood probe compares each true answer with this many same-relation distractors.
LIKELIHOOD_DISTRACTOR_COUNT = 3
# Keep likelihood batches small because model logits retain the full vocabulary dimension.
LIKELIHOOD_BATCH_SIZE = 4
MAX_EDIT_BANK = 100
ORACLE_STRENGTHS = (0.25, 0.5, 0.75, 1.0)
BATCH_SIZE = 4
MAX_NEW_TOKENS = 24
REPETITION_PENALTY = 1.5
CONTEXT_TEMPLATES = ('new_fact', 'qa')
INCLUDE_WRONG_CONTEXT = True

# The hidden-state diagnostic deliberately starts in external-only mode.
EXTERNAL_ONLY = True

print(f'data={DATA_PATH}')
print(f'oracle_output={ORACLE_OUTPUT_PATH}')
print(f'context_output={CONTEXT_OUTPUT_PATH}')
print(f'natural_output={NATURAL_OUTPUT_PATH}')
print(f'prerequisite_output={PREREQUISITE_OUTPUT_PATH}')
print(f'prerequisite_screen_output={PREREQUISITE_SCREEN_OUTPUT_PATH}')
print(f'prerequisite_likelihood_output={PREREQUISITE_LIKELIHOOD_OUTPUT_PATH}')
print(f'natural_likelihood_output={NATURAL_LIKELIHOOD_OUTPUT_PATH}')
print(f'query_limit_per_stratum={QUERY_LIMIT_PER_STRATUM}, bank_cap={MAX_EDIT_BANK}')
print(f'prerequisite_screen_limit_per_stratum={PREREQUISITE_SCREEN_LIMIT_PER_STRATUM}')

In [ ]:
# Uncomment the action you want to run.
# run_oracle_experiment()
# run_context_experiment()
# run_natural_context_experiment()
# run_prerequisite_experiment()
# run_prerequisite_screening()
# run_prerequisite_likelihood_experiment()
# run_likelihood_screened_natural_experiment()